# Corrected Full MEI Checks (Workflow 1)

This is a Workflow 1 maintainer notebook for already-combined, xml:id-clean MEI files. It uses the reusable checks installed with `camat`; only the edited MEI files and generated reports remain in the edition repository. By default it performs check-only MEI 5.1 CMN RELAX NG, publication-profile, facsimile-topology, musical-consistency, and Verovio passes, then summarizes the findings. Enable a `CLEAN_*` or `FIX_*` flag only when you intentionally want to rewrite the selected files.

In [ ]:
from pathlib import Path
import pandas as pd

from camat import (
    MEI_CMN_51_SCHEMA,
    VEROVIO_REPORT_COLUMNS,
    apply_safe_cleanup,
    convert_harm_startid_to_tstamp,
    link_pb_to_surface,
    load_report,
    normalize_single_layer_number_copies,
    report_summary,
    resolve_mei_inputs,
    run_checker,
    run_relaxng_validation,
    run_verovio_warning_check,
    source_snippet,
)

# Set this to the directory against which relative input paths and report paths are resolved.
ROOT = Path.cwd()

FULL_MEI_INPUTS = [
    # "11_buxtehude_sonatas_final/buxtehude_op1_01_sonata_f_major_corr.mei",
    # "11_buxtehude_sonatas_final/buxtehude_op1_02_sonata_g_major_corr.mei",
    # "11_buxtehude_sonatas_final/buxtehude_op1_03_sonata_a_minor_corr.mei",
    # "11_buxtehude_sonatas_final/buxtehude_op1_04_sonata_b_flat_major_corr.mei",
    # "11_buxtehude_sonatas_final/buxtehude_op1_05_sonata_c_major_corr.mei",
    # "11_buxtehude_sonatas_final/buxtehude_op1_06_sonata_d_minor_corr.mei",
    # "11_buxtehude_sonatas_final/buxtehude_op1_07_sonata_e_minor_corr.mei",
    # "11_buxtehude_sonatas_final/buxtehude_op2_02_sonata_d_major_corr.mei",
    # "11_buxtehude_sonatas_final/buxtehude_op2_03_sonata_g_minor_corr.mei",
    # "11_buxtehude_sonatas_final/buxtehude_op2_05_sonata_a_major_corr.mei",
    # "11_buxtehude_sonatas_final/buxtehude_op2_06_sonata_e_major_corr.mei",
    # "11_buxtehude_sonatas_final/buxtehude_app_01_suite_to_op1_04_corr.mei"
]

CHECK_PPQ = True  # Compare optional @dur.ppq against written @dur/@dots.
CHECK_RELAXNG = True  # Validate selected files against the pinned MEI 5.1 CMN schema.
RELAXNG_SCHEMA = MEI_CMN_51_SCHEMA  # Pinned schema distributed with camat.
CHECK_PUBLICATION_PROFILE = True  # Check header, identifiers, and facsimile page topology.
CLEAN_PPQ = False  # Rewrite files: strip @ppq and @dur.ppq.
NORMALIZE_SINGLE_LAYER_NUMBERS = False  # Rewrite single-layer @n copies before checking.
NORMALIZED_LAYER_SUFFIX = ""  # Suffix for those copies; empty writes in place.
CLEAN_ACCID = False  # Rewrite files: remove empty <accid/> children and duplicate note-level accidentals.
CLEAN_ACCID_GES = False  # Rewrite files: strip leftover MusicXML @accid.ges (MIDI vs render).
FIX_TIES_SAFE = False  # Rewrite files: conservative, safe tie cleanup only.
CHECK_FB_TSTAMP = True  # Report figured-bass <harm> that still use @startid.
FIX_FB_TSTAMP = False  # Rewrite files: convert those harms to @tstamp + @staff.
CHECK_PB_FACS = True  # Check <pb> @facs against the following measure's surface.
FIX_PB_FACS = False  # Rewrite files: fill missing <pb> @facs links.
OVERWRITE_PB_FACS = False  # Also overwrite mismatched <pb> @facs when FIX_PB_FACS is on.
CHECK_VEROVIO = True  # Load with Verovio and collect warning/error logs.
VEROVIO_RENDER_PAGES = True  # Also render every page; slower, catches layout-only issues.

FULL_MEI_FILES = resolve_mei_inputs(FULL_MEI_INPUTS, ROOT)
if not FULL_MEI_FILES:
    raise FileNotFoundError("Set ROOT and add at least one MEI file or directory to FULL_MEI_INPUTS")

ACTIVE_FULL_MEI_FILES = FULL_MEI_FILES
if NORMALIZE_SINGLE_LAYER_NUMBERS:
    normalize_result = normalize_single_layer_number_copies(
        FULL_MEI_FILES,
        suffix=NORMALIZED_LAYER_SUFFIX,
    )
    ACTIVE_FULL_MEI_FILES = normalize_result.files
    print(normalize_result.message)

cleanup_result = apply_safe_cleanup(
    ACTIVE_FULL_MEI_FILES,
    root=ROOT,
    clean_ppq=CLEAN_PPQ,
    clean_accid=CLEAN_ACCID,
    clean_accid_ges=CLEAN_ACCID_GES,
    fix_ties=FIX_TIES_SAFE,
)
CLEANUP_ROWS = cleanup_result.rows
if CLEAN_PPQ or CLEAN_ACCID or CLEAN_ACCID_GES or FIX_TIES_SAFE:
    print(cleanup_result.message)

FB_TSTAMP_ROWS = []
if CHECK_FB_TSTAMP:
    for path in ACTIVE_FULL_MEI_FILES:
        fb_result = convert_harm_startid_to_tstamp(path, apply=FIX_FB_TSTAMP)
        shown_path = path.relative_to(ROOT) if path.is_relative_to(ROOT) else path
        action_text = "converted" if FIX_FB_TSTAMP else "would convert"
        print(
            f"{shown_path}: figured-bass tstamp check: "
            f"{fb_result.converted} {action_text}, "
            f"{fb_result.skipped} skipped, {fb_result.unresolved} unresolved."
        )

        for row in fb_result.rows:
            if row.status == "convert" and FIX_FB_TSTAMP:
                severity = "info"
                message = "Figured-bass harm was converted from startid to tstamp+staff."
            elif row.status == "convert":
                severity = "warning"
                message = "Figured-bass harm uses startid and can be converted to tstamp+staff."
            elif row.status == "skipped_cross_measure":
                severity = "warning"
                message = "Figured-bass startid points to a timed event in a different measure; inspect manually."
            elif row.status == "unresolved":
                severity = "error"
                message = "Figured-bass startid target is missing or not a timed event."
            else:
                continue

            FB_TSTAMP_ROWS.append({
                "severity": severity,
                "category": "figured_bass",
                "check": "fb_startid_to_tstamp",
                "file": row.file,
                "line": row.line or "",
                "element": "harm",
                "measure_n": row.measure_n,
                "staff_n": row.staff,
                "layer_n": "",
                "xml_id": "",
                "message": message,
                "expected": f'tstamp="{row.tstamp}" staff="{row.staff}"' if row.tstamp else "",
                "actual": f'startid="#{row.startid}"',
                "context": row.status,
            })

PB_FACS_ROWS = []
if CHECK_PB_FACS:
    for path in ACTIVE_FULL_MEI_FILES:
        pb_result = link_pb_to_surface(path, apply=FIX_PB_FACS, overwrite=OVERWRITE_PB_FACS)
        shown_path = path.relative_to(ROOT) if path.is_relative_to(ROOT) else path
        action_text = "updated" if FIX_PB_FACS else "would update"
        print(
            f"{shown_path}: page-break facsimile check: "
            f"{pb_result.updates} {action_text}, "
            f"{pb_result.mismatches} mismatched, {pb_result.unresolved} unresolved."
        )

        for row in pb_result.rows:
            if row.status == "ok":
                continue
            if row.status == "missing" and FIX_PB_FACS:
                severity = "info"
                message = "Page break @facs was linked to the source surface."
            elif row.status == "missing":
                severity = "warning"
                message = "Page break has no @facs link to the source surface."
            elif row.status == "mismatch" and FIX_PB_FACS and OVERWRITE_PB_FACS:
                severity = "info"
                message = "Page break @facs was updated to match the source surface."
            elif row.status == "mismatch":
                severity = "warning"
                message = "Page break @facs does not match the following measure's source surface."
            elif row.status == "unresolved":
                severity = "error"
                message = row.message
            else:
                continue

            PB_FACS_ROWS.append({
                "severity": severity,
                "category": "facsimile",
                "check": "pb_facs_surface_link",
                "file": row.file,
                "line": row.line or "",
                "element": "pb",
                "measure_n": row.measure_n,
                "staff_n": "",
                "layer_n": "",
                "xml_id": row.pb_id,
                "message": message,
                "expected": f'facs="#{row.expected_facs}"' if row.expected_facs else "",
                "actual": f'facs="#{row.current_facs}"' if row.current_facs else "",
                "context": row.status,
            })

SCHEMA_ROWS = []
if CHECK_RELAXNG:
    schema_result = run_relaxng_validation(
        ACTIVE_FULL_MEI_FILES,
        schema=RELAXNG_SCHEMA,
        root=ROOT,
    )
    SCHEMA_ROWS = schema_result.rows
    print(
        f"MEI 5.1 CMN RELAX NG: {schema_result.files_checked} file(s), "
        f"{len(SCHEMA_ROWS)} validation error(s), schema={schema_result.schema.name}."
    )


REPORT_EXTRA_ROWS = CLEANUP_ROWS + FB_TSTAMP_ROWS + PB_FACS_ROWS + SCHEMA_ROWS

REPORT_TARGETS = {
    path: {
        "csv": path.with_name(f"{path.stem}_consistency_report.csv"),
        "json": path.with_name(f"{path.stem}_consistency_report.json"),
    }
    for path in ACTIVE_FULL_MEI_FILES
}

print(f"Selected {len(ACTIVE_FULL_MEI_FILES)} corrected full MEI file(s) for checking.")
for path in ACTIVE_FULL_MEI_FILES:
    report_path = REPORT_TARGETS[path]["csv"]
    shown_path = path.relative_to(ROOT) if path.is_relative_to(ROOT) else path
    shown_report = report_path.relative_to(ROOT) if report_path.is_relative_to(ROOT) else report_path
    print(f"{shown_path} -> {shown_report}")


In [ ]:
checker_results = []
report_frames = []

for path in ACTIVE_FULL_MEI_FILES:
    csv_out = REPORT_TARGETS[path]["csv"]
    checker_results.append(
        run_checker(
            [path],
            root=ROOT,
            csv_out=csv_out,
            json_out=None,
            check_ppq=CHECK_PPQ,
            publication_profile=CHECK_PUBLICATION_PROFILE,
        )
    )
    file_df = load_report(csv_out)
    cleanup_file_df = pd.DataFrame(
        [row for row in REPORT_EXTRA_ROWS if row["file"] == (path.relative_to(ROOT).as_posix() if path.is_relative_to(ROOT) else path.as_posix())],
        columns=VEROVIO_REPORT_COLUMNS,
    )
    if not cleanup_file_df.empty:
        file_df = pd.concat([file_df, cleanup_file_df], ignore_index=True)
        file_df.to_csv(csv_out, index=False)
    report_frames.append(file_df)
    shown_report = csv_out.relative_to(ROOT) if csv_out.is_relative_to(ROOT) else csv_out
    print(f"Loaded {len(file_df)} finding(s) from {shown_report}.")

df = pd.concat(report_frames, ignore_index=True) if report_frames else pd.DataFrame()
print(f"Loaded {len(df)} total finding(s) from {len(report_frames)} per-file report(s).")
report_summary(df)


In [ ]:
if CHECK_VEROVIO:
    verovio_results = []
    verovio_frames = []

    for path in ACTIVE_FULL_MEI_FILES:
        csv_out = REPORT_TARGETS[path]["csv"]
        verovio_result = run_verovio_warning_check(
            [path],
            root=ROOT,
            render_pages=VEROVIO_RENDER_PAGES,
        )
        verovio_results.append(verovio_result)
        verovio_file_df = verovio_result.to_dataframe()
        verovio_frames.append(verovio_file_df)

        file_df = load_report(csv_out)
        file_df = file_df[~file_df["category"].eq("verovio")].copy()
        if not verovio_file_df.empty:
            file_df = pd.concat([file_df, verovio_file_df], ignore_index=True)
        file_df.to_csv(csv_out, index=False)

        shown_report = csv_out.relative_to(ROOT) if csv_out.is_relative_to(ROOT) else csv_out
        print(
            f"{shown_report}: Verovio {verovio_result.version}; "
            f"{len(verovio_file_df)} warning/error log row(s); "
            f"{len(file_df)} total report row(s)."
        )

    df = pd.concat([load_report(REPORT_TARGETS[path]["csv"]) for path in ACTIVE_FULL_MEI_FILES], ignore_index=True)
    verovio_df = pd.concat(verovio_frames, ignore_index=True) if verovio_frames else pd.DataFrame()
    print(f"Report set now contains {len(df)} total row(s) across {len(ACTIVE_FULL_MEI_FILES)} file(s).")
    verovio_df.head(100)
else:
    verovio_results = []
    verovio_df = None
    print("Verovio warning check disabled.")


In [ ]:
errors_df = df[df["severity"].eq("error")].copy()
print(f"Error-level findings: {len(errors_df)}")
errors_df[[
    "severity", "category", "check", "file", "line", "measure_n", "staff_n", "layer_n", "xml_id", "message", "expected", "actual", "context"
]].head(100)


In [ ]:
warnings_df = df[df["severity"].eq("warning")].copy()
print(f"Warning-level findings: {len(warnings_df)}")
warnings_df[[
    "severity", "category", "check", "file", "line", "measure_n", "staff_n", "layer_n", "xml_id", "message", "expected", "actual", "context"
]].head(100)


In [ ]:
infos_df = df[df["severity"].eq("info")].copy()
print(f"Info-level findings: {len(infos_df)}")
infos_df[[
    "severity", "category", "check", "file", "line", "measure_n", "staff_n", "layer_n", "xml_id", "message", "expected", "actual", "context"
]].head(100)


In [ ]:
# Change the index to inspect a finding in context.
ROW_INDEX = None

if ROW_INDEX is not None:
    source_snippet(df.loc[ROW_INDEX], root=ROOT, active_files=ACTIVE_FULL_MEI_FILES, radius=4)
else:
    print("Set ROW_INDEX to a row number from df, errors_df, warnings_df, or infos_df to inspect source context.")
